# Zadanie 2: Predyckja wieku małży (Abalone)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold
from sklearn.svm import SVR
from sklearn.linear_model import Ridge

from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import optuna

c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("../Data/abalone.csv")

data.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   object 
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4177 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


In [6]:
y = data["Rings"]
X = data.drop(columns=["Rings"])
X = pd.get_dummies(X, drop_first=True,columns=["Sex"],prefix="Sex")
X.head()



,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Sex_I,Sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,False,True
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,False,True
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,False,False
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,False,True
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,True,False


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3)

In [16]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [17]:
def create_best_model(X_train,y_train,model_name:str):
    def objective(trial:optuna.trial.Trial):
        n_features = trial.suggest_int("n_features",1,X_train.shape[1])

        if model_name == "SVR":
            C = trial.suggest_float("C",0.01,1.0,log=True)
            epsilon = trial.suggest_float("epsilon",0.01,1.0,log=True)
            kernel = trial.suggest_categorical("kernel",["rbf","linear"])

            if kernel == "rbf":
                gamma = trial.suggest_categorical("gamma",["scale","auto"])
                model = SVR(C=C,epsilon=epsilon,kernel=kernel,gamma=gamma)
                svr_estimator = SVR(kernel="linear")
            else:
                svr_estimator = None
                model = SVR(C=C,epsilon=epsilon,kernel=kernel)
        
        elif model_name == "Ridge":
            alpha = trial.suggest_float("alpha",0.01,100.0,log=True)
            model = Ridge(alpha=alpha)
        
        pipeline = Pipeline([
            ("scaler",StandardScaler()),
            ("feature_selection",RFE(estimator=model if svr_estimator == None else svr_estimator ,n_features_to_select=n_features)),
            ("model",model)
        ])

        cv = StratifiedKFold(n_splits=10,shuffle=True)

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="neg_mean_squared_error"
        )

        return -np.mean(scores)
    
    study = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler())
    study.optimize(objective,n_trials=25)
    return study.best_params, study.best_value

In [18]:
best_SVR_params, best_SVR_score = create_best_model(X_train,y_train,"SVR")
print("Best SVR params:", best_SVR_params)
print("Best SVR CV MSE:", best_SVR_score)

best_Ridge_params, best_Ridge_score = create_best_model(X_train,y_train,"Ridge")
print("Best Ridge params:", best_Ridge_params)
print("Best Ridge CV MSE:", best_Ridge_score)

c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(
c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(
c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(
c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(
c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 

Best SVR params: {'n_features': 9, 'C': 0.991430907269098, 'epsilon': 0.17497988464771766, 'kernel': 'linear'}
Best SVR CV MSE: 5.245151750654619


UnboundLocalError: cannot access local variable 'svr_estimator' where it is not associated with a value